In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix,precision_score, recall_score,f1_score,roc_curve, roc_auc_score, ConfusionMatrixDisplay, make_scorer
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
import time
import copy
import os
from collections import Counter
from sklearn.metrics import log_loss
import pickle

import import_ipynb
import importlib
import functions as fc
importlib.reload(fc)

print(os.getcwd())

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

c:\Users\ameli\OneDrive\Studium\TUWien\SS2026\PrivacyInML\Exercise2


In [2]:
def evaluate_true_class_confidence(input_confidence_scores, y_train, y_test):

    true_labels = np.concatenate([y_train, y_test])
    class_order = np.array([0, 1])

    membership = np.concatenate([np.ones(len(y_train)),np.zeros(len(y_test))])

    all_mean = []
    member_mean = []
    nonmember_mean = []
    member_loss_mean = []
    nonmember_loss_mean = []
    member_entropy_mean = []
    nonmember_entropy_mean = []

    for key, scores in input_confidence_scores.items():

        true_class_indices = np.array([np.where(class_order == label)[0][0]for label in true_labels])
        true_confidences = scores[np.arange(len(true_labels)),true_class_indices]
        member_confidence = true_confidences[:len(y_train)]
        nonmember_confidence = true_confidences[len(y_train):]

        ##### LOSS #####
        loss = -np.log(np.clip(true_confidences, 1e-10, 1.0))
        member_loss = loss[:len(y_train)]
        nonmember_loss = loss[len(y_train):]

        # Higher confidence = more likely member
        auc = roc_auc_score(membership,true_confidences)
        # Lower loss = more likely member
        auc_loss = roc_auc_score(membership,-loss)

        ##### ENTROPY #####
        entropy = -np.sum(scores * np.log(np.clip(scores, 1e-10, 1.0)),axis=1)
        member_entropy = entropy[:len(y_train)]
        nonmember_entropy = entropy[len(y_train):]
        # Lower entropy = more confident prediction
        auc_entropy = roc_auc_score(membership,-entropy)

        ##### RESULTS #####
        print('-----------------')
        print(f"Model {key}")
        print('-----------------')

        print("Confidence:")
        print("  Overall mean:", true_confidences.mean())
        print("  Member mean:", member_confidence.mean())
        print("  Non-member mean:", nonmember_confidence.mean())
        print("  Confidence gap:",
              member_confidence.mean() - nonmember_confidence.mean())
        print("  MIA AUC:", auc)

        print()
        print("Loss:")
        print("  Overall mean:", loss.mean())
        print("  Member mean:", member_loss.mean())
        print("  Non-member mean:", nonmember_loss.mean())
        print("  Loss gap:",
              nonmember_loss.mean() - member_loss.mean())
        print("  MIA AUC:", auc_loss)

        print()
        print("Entropy:")
        print("  Overall mean:", entropy.mean())
        print("  Member mean:", member_entropy.mean())
        print("  Non-member mean:", nonmember_entropy.mean())
        print("  Entropy gap:",
              nonmember_entropy.mean() - member_entropy.mean())
        print("  MIA AUC:", auc_entropy)

        print()
        print("Confidence:")
        print("  Minimum:", true_confidences.min())
        print("  Maximum:", true_confidences.max())

        print('=========================================\n')

        all_mean.append(true_confidences.mean())
        member_mean.append(member_confidence.mean())
        nonmember_mean.append(nonmember_confidence.mean())
        member_loss_mean.append(member_loss.mean())
        nonmember_loss_mean.append(nonmember_loss.mean())
        member_entropy_mean.append(member_entropy.mean())
        nonmember_entropy_mean.append(nonmember_entropy.mean())


    # ======================================================
    # ACROSS ALL MODELS
    # ======================================================
    print("=== Across all models ===")
    print("Average overall confidence:", np.mean(all_mean))
    print("Average member confidence:", np.mean(member_mean))
    print("Average non-member confidence:", np.mean(nonmember_mean))
    print()
    print("Average member loss:", np.mean(member_loss_mean))
    print("Average non-member loss:", np.mean(nonmember_loss_mean))
    print()
    print("Average member entropy:", np.mean(member_entropy_mean))
    print("Average non-member entropy:", np.mean(nonmember_entropy_mean))

In [3]:
def evaluate_true_max_confidence(input_confidence_scores, y_train, y_test):

    true_labels = np.concatenate([y_train, y_test])
    class_order = np.array([0, 1])

    membership = np.concatenate([np.ones(len(y_train)),np.zeros(len(y_test))])

    all_mean = []
    member_mean = []
    nonmember_mean = []
    member_loss_mean = []
    nonmember_loss_mean = []
    member_entropy_mean = []
    nonmember_entropy_mean = []

    for key, scores in input_confidence_scores.items():

        true_confidences = scores.max(axis=1)
        member_confidence = true_confidences[:len(y_train)]
        nonmember_confidence = true_confidences[len(y_train):]

        ##### LOSS #####
        loss = -np.log(np.clip(true_confidences, 1e-10, 1.0))
        member_loss = loss[:len(y_train)]
        nonmember_loss = loss[len(y_train):]

        # Higher confidence = more likely member
        auc = roc_auc_score(membership,true_confidences)
        # Lower loss = more likely member
        auc_loss = roc_auc_score(membership,-loss)

        ##### ENTROPY #####
        entropy = -np.sum(scores * np.log(np.clip(scores, 1e-10, 1.0)),axis=1)
        member_entropy = entropy[:len(y_train)]
        nonmember_entropy = entropy[len(y_train):]
        # Lower entropy = more confident prediction
        auc_entropy = roc_auc_score(membership,-entropy)

        ##### RESULTS #####
        print('-----------------')
        print(f"Model {key}")
        print('-----------------')

        print("Confidence:")
        print("  Overall mean:", true_confidences.mean())
        print("  Member mean:", member_confidence.mean())
        print("  Non-member mean:", nonmember_confidence.mean())
        print("  Confidence gap:",
              member_confidence.mean() - nonmember_confidence.mean())
        print("  MIA AUC:", auc)

        print()
        print("Loss:")
        print("  Overall mean:", loss.mean())
        print("  Member mean:", member_loss.mean())
        print("  Non-member mean:", nonmember_loss.mean())
        print("  Loss gap:",
              nonmember_loss.mean() - member_loss.mean())
        print("  MIA AUC:", auc_loss)

        print()
        print("Entropy:")
        print("  Overall mean:", entropy.mean())
        print("  Member mean:", member_entropy.mean())
        print("  Non-member mean:", nonmember_entropy.mean())
        print("  Entropy gap:",
              nonmember_entropy.mean() - member_entropy.mean())
        print("  MIA AUC:", auc_entropy)

        print()
        print("Confidence:")
        print("  Minimum:", true_confidences.min())
        print("  Maximum:", true_confidences.max())

        print('=========================================\n')

        all_mean.append(true_confidences.mean())
        member_mean.append(member_confidence.mean())
        nonmember_mean.append(nonmember_confidence.mean())
        member_loss_mean.append(member_loss.mean())
        nonmember_loss_mean.append(nonmember_loss.mean())
        member_entropy_mean.append(member_entropy.mean())
        nonmember_entropy_mean.append(nonmember_entropy.mean())

    # ======================================================
    # ACROSS ALL MODELS
    # ======================================================
    print("=== Across all models ===")
    print("Average overall confidence:", np.mean(all_mean))
    print("Average member confidence:", np.mean(member_mean))
    print("Average non-member confidence:", np.mean(nonmember_mean))
    print()
    print("Average member loss:", np.mean(member_loss_mean))
    print("Average non-member loss:", np.mean(nonmember_loss_mean))
    print()
    print("Average member entropy:", np.mean(member_entropy_mean))
    print("Average non-member entropy:", np.mean(nonmember_entropy_mean))

In [4]:
def evaluate_threshold_attack(
    input_confidence_scores,
    y_train,
    y_test,
    threshold
):

    membership = np.concatenate([np.ones(len(y_train)),np.zeros(len(y_test))])
    for key, scores in input_confidence_scores.items():

        # Use the highest predicted probability
        confidence = scores.max(axis=1)

        # Predict membership based on threshold
        attack_labels = np.array([1 if conf > threshold else 0 for conf in confidence])

        print('-----------------')
        print(f"Model {key}")
        print('-----------------')

        print("Threshold:", threshold)

        print("Predicted members:",np.sum(attack_labels == 1))
        print("Predicted non-members:",np.sum(attack_labels == 0))
        print("Actual members:",np.sum(membership == 1))
        print("Actual non-members:",np.sum(membership == 0))

        print("Accuracy:",accuracy_score(membership, attack_labels))
        print("Confusion matrix:")

        cm = confusion_matrix(membership,attack_labels)
        print(cm)
        print('=========================================\n')

In [ ]:
diabetes_train = pd.read_csv("../datasets/diabetes_train.csv")
diabetes_test = pd.read_csv("../datasets/diabetes_test.csv")

In [6]:
X_train = diabetes_train.drop('Outcome', axis = 1)
X_test = diabetes_test.drop('Outcome', axis = 1)
y_train = diabetes_train['Outcome']
y_test = diabetes_test['Outcome']

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

----------------------------------------------------------------------------------------------
-------------------------------------- INPUT PERTUBATION -------------------------------------
----------------------------------------------------------------------------------------------

In [7]:
with open("input_confidence_scores_diabetes.pkl", "rb") as f:
    input_confidence_scores = pickle.load(f)

In [8]:
evaluate_true_class_confidence(input_confidence_scores, y_train, y_test)
evaluate_true_max_confidence(input_confidence_scores, y_train, y_test)

-----------------
Model 0.01
-----------------
Confidence:
  Overall mean: 0.6470910711441783
  Member mean: 0.6497436128045657
  Non-member mean: 0.6365153530956204
  Confidence gap: 0.013228259708945367
  MIA AUC: 0.5347413173146072

Loss:
  Overall mean: 2.4811887484311987
  Member mean: 2.4716749623557672
  Non-member mean: 2.519120337069608
  Loss gap: 0.04744537471384058
  MIA AUC: 0.5347413173146072

Entropy:
  Overall mean: 0.0854203539073159
  Member mean: 0.08636968240069912
  Non-member mean: 0.08163536887525553
  Entropy gap: -0.004734313525443581
  MIA AUC: 0.5325521384153307

Confidence:
  Minimum: 5.508038469770327e-12
  Maximum: 1.0

-----------------
Model 0.1
-----------------
Confidence:
  Overall mean: 0.6441507315788272
  Member mean: 0.6461264501963
  Non-member mean: 0.6362735157922793
  Confidence gap: 0.009852934404020686
  MIA AUC: 0.5076039595583569

Loss:
  Overall mean: 3.9547878173530293
  Member mean: 3.91607899227056
  Non-member mean: 4.109120405668848


-----------------
Model 30
-----------------
Confidence:
  Overall mean: 0.3489583333333333
  Member mean: 0.3469055374592834
  Non-member mean: 0.35714285714285715
  Confidence gap: -0.010237319683573776
  MIA AUC: 0.49488134015821306

Loss:
  Overall mean: 14.99078836584665
  Member mean: 15.038055737632122
  Non-member mean: 14.802332740676007
  Loss gap: -0.23572299695611498
  MIA AUC: 0.49488134015821306

Entropy:
  Overall mean: 0.0
  Member mean: 0.0
  Non-member mean: 0.0
  Entropy gap: 0.0
  MIA AUC: 0.5

Confidence:
  Minimum: 0.0
  Maximum: 1.0

-----------------
Model 50
-----------------
Confidence:
  Overall mean: 0.3489583333333333
  Member mean: 0.3469055374592834
  Non-member mean: 0.35714285714285715
  Confidence gap: -0.010237319683573776
  MIA AUC: 0.49488134015821306

Loss:
  Overall mean: 14.99078836584665
  Member mean: 15.038055737632122
  Non-member mean: 14.802332740676007
  Loss gap: -0.23572299695611498
  MIA AUC: 0.49488134015821306

Entropy:
  Overall mean

In [9]:
evaluate_threshold_attack(input_confidence_scores,y_train,y_test,threshold=0.970489258446794)

-----------------
Model 0.01
-----------------
Threshold: 0.970489258446794
Predicted members: 631
Predicted non-members: 137
Actual members: 614
Actual non-members: 154
Accuracy: 0.6888020833333334
Confusion matrix:
[[ 26 128]
 [111 503]]

-----------------
Model 0.1
-----------------
Threshold: 0.970489258446794
Predicted members: 729
Predicted non-members: 39
Actual members: 614
Actual non-members: 154
Accuracy: 0.7669270833333334
Confusion matrix:
[[  7 147]
 [ 32 582]]

-----------------
Model 1
-----------------
Threshold: 0.970489258446794
Predicted members: 55
Predicted non-members: 713
Actual members: 614
Actual non-members: 154
Accuracy: 0.24609375
Confusion matrix:
[[144  10]
 [569  45]]

-----------------
Model 5
-----------------
Threshold: 0.970489258446794
Predicted members: 768
Predicted non-members: 0
Actual members: 614
Actual non-members: 154
Accuracy: 0.7994791666666666
Confusion matrix:
[[  0 154]
 [  0 614]]

-----------------
Model 10
-----------------
Threshold:

----------------------------------------------------------------------------------------------
------------------------------------- OUTPUT PERTUBATION -------------------------------------
----------------------------------------------------------------------------------------------

In [10]:
with open("output_confidence_scores_diabetes.pkl", "rb") as f:
    output_confidence_scores = pickle.load(f)

In [11]:
evaluate_true_class_confidence(output_confidence_scores, y_train, y_test)
evaluate_true_max_confidence(output_confidence_scores, y_train, y_test)

-----------------
Model 0.01
-----------------
Confidence:
  Overall mean: 0.34520111388809366
  Member mean: 0.3438346180229085
  Non-member mean: 0.35064935064928626
  Confidence gap: -0.006814732626377773
  MIA AUC: 0.4997726215152925

Loss:
  Overall mean: 14.985443356456555
  Member mean: 14.993868737401593
  Non-member mean: 14.951851253208151
  Loss gap: -0.0420174841934422
  MIA AUC: 0.5002591057151318

Entropy:
  Overall mean: 0.0006076041572903007
  Member mean: 0.0007599999878837865
  Non-member mean: 1.5474414838115956e-12
  Entropy gap: -0.000759999986336345
  MIA AUC: 0.497377215618258

Confidence:
  Minimum: 0.0
  Maximum: 1.0

-----------------
Model 0.1
-----------------
Confidence:
  Overall mean: 0.34318798282242097
  Member mean: 0.3432963398839768
  Non-member mean: 0.3427559618107634
  Confidence gap: 0.0005403780732133878
  MIA AUC: 0.5007138626845468

Loss:
  Overall mean: 14.723856664273903
  Member mean: 14.737556160421295
  Non-member mean: 14.669236595218713

-----------------
Model 10
-----------------
Confidence:
  Overall mean: 0.3489583333333333
  Member mean: 0.3469055374592834
  Non-member mean: 0.35714285714285715
  Confidence gap: -0.010237319683573776
  MIA AUC: 0.49488134015821306

Loss:
  Overall mean: 14.99078836584665
  Member mean: 15.038055737632122
  Non-member mean: 14.802332740676007
  Loss gap: -0.23572299695611498
  MIA AUC: 0.49488134015821306

Entropy:
  Overall mean: 0.0
  Member mean: 0.0
  Non-member mean: 0.0
  Entropy gap: 0.0
  MIA AUC: 0.5

Confidence:
  Minimum: 0.0
  Maximum: 1.0

-----------------
Model 30
-----------------
Confidence:
  Overall mean: 0.3489583333333333
  Member mean: 0.3469055374592834
  Non-member mean: 0.35714285714285715
  Confidence gap: -0.010237319683573776
  MIA AUC: 0.49488134015821306

Loss:
  Overall mean: 14.99078836584665
  Member mean: 15.038055737632122
  Non-member mean: 14.802332740676007
  Loss gap: -0.23572299695611498
  MIA AUC: 0.49488134015821306

Entropy:
  Overall mean

In [12]:
evaluate_threshold_attack(output_confidence_scores,y_train,y_test,threshold=0.970489258446794)

-----------------
Model 0.01
-----------------
Threshold: 0.970489258446794
Predicted members: 767
Predicted non-members: 1
Actual members: 614
Actual non-members: 154
Accuracy: 0.7981770833333334
Confusion matrix:
[[  0 154]
 [  1 613]]

-----------------
Model 0.1
-----------------
Threshold: 0.970489258446794
Predicted members: 760
Predicted non-members: 8
Actual members: 614
Actual non-members: 154
Accuracy: 0.7942708333333334
Confusion matrix:
[[  2 152]
 [  6 608]]

-----------------
Model 1
-----------------
Threshold: 0.970489258446794
Predicted members: 768
Predicted non-members: 0
Actual members: 614
Actual non-members: 154
Accuracy: 0.7994791666666666
Confusion matrix:
[[  0 154]
 [  0 614]]

-----------------
Model 5
-----------------
Threshold: 0.970489258446794
Predicted members: 768
Predicted non-members: 0
Actual members: 614
Actual non-members: 154
Accuracy: 0.7994791666666666
Confusion matrix:
[[  0 154]
 [  0 614]]

-----------------
Model 10
-----------------
Thresh

----------------------------------------------------------------------------------------------
------------------------------------ INTERNAL PERTUBATION ------------------------------------
----------------------------------------------------------------------------------------------

In [13]:
with open("internal_confidence_scores_diabetes.pkl", "rb") as f:
    internal_confidence_scores = pickle.load(f)

In [14]:
evaluate_true_class_confidence(internal_confidence_scores, y_train, y_test)
evaluate_true_max_confidence(internal_confidence_scores, y_train, y_test)

-----------------
Model 0.01
-----------------
Confidence:
  Overall mean: 0.5445220107571948
  Member mean: 0.5546509494474962
  Non-member mean: 0.5041378006543048
  Confidence gap: 0.050513148793191354
  MIA AUC: 0.5167784170227168

Loss:
  Overall mean: 7.202112310657149
  Member mean: 7.1046656707428575
  Non-member mean: 7.59063332953621
  Loss gap: 0.4859676587933528
  MIA AUC: 0.5176086128854859

Entropy:
  Overall mean: 0.04510027069903209
  Member mean: 0.04150446660158176
  Non-member mean: 0.05943678833432105
  Entropy gap: 0.017932321732739287
  MIA AUC: 0.5142931173061467

Confidence:
  Minimum: 0.0
  Maximum: 1.0

-----------------
Model 0.1
-----------------
Confidence:
  Overall mean: 0.5587569872708081
  Member mean: 0.5671002116644376
  Non-member mean: 0.5254924432598437
  Confidence gap: 0.0416077684045939
  MIA AUC: 0.5176297643724354

Loss:
  Overall mean: 6.645699992349887
  Member mean: 6.567683325393152
  Non-member mean: 6.956753456709864
  Loss gap: 0.389070

In [15]:
evaluate_threshold_attack(internal_confidence_scores,y_train,y_test,threshold=0.970489258446794)

-----------------
Model 0.01
-----------------
Threshold: 0.970489258446794
Predicted members: 695
Predicted non-members: 73
Actual members: 614
Actual non-members: 154
Accuracy: 0.75390625
Confusion matrix:
[[ 19 135]
 [ 54 560]]

-----------------
Model 0.1
-----------------
Threshold: 0.970489258446794
Predicted members: 692
Predicted non-members: 76
Actual members: 614
Actual non-members: 154
Accuracy: 0.7473958333333334
Confusion matrix:
[[ 18 136]
 [ 58 556]]

-----------------
Model 1
-----------------
Threshold: 0.970489258446794
Predicted members: 586
Predicted non-members: 182
Actual members: 614
Actual non-members: 154
Accuracy: 0.65625
Confusion matrix:
[[ 36 118]
 [146 468]]

-----------------
Model 5
-----------------
Threshold: 0.970489258446794
Predicted members: 36
Predicted non-members: 732
Actual members: 614
Actual non-members: 154
Accuracy: 0.21875
Confusion matrix:
[[143  11]
 [589  25]]

-----------------
Model 10
-----------------
Threshold: 0.970489258446794
Pr